# Capstone: a real dataset, honestly explored

MichAl Academy, lesson 1.11.

Three thousand URLs with features already extracted and a `label` column. It
looks ready to model on.

**Five things are wrong with it and none of them are labelled.** Find them before
you would have trained anything.

Work the tasks in order. Each has a check cell that tells you when you have it.
Answers are at the bottom, and using them early wastes the exercise.

Budget about ninety minutes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def build_export():
    """The export you were handed. Do not read this too closely yet."""
    rng = np.random.default_rng(20260909)
    N = 2900

    tlds = rng.choice(["com", "co", "net", "tk", "xyz", "org"], N, p=[.42, .14, .10, .14, .10, .10])
    phish_p = np.where(np.isin(tlds, ["tk", "xyz"]), 0.22, 0.02)
    label = (rng.random(N) < phish_p).astype(int)

    url_length  = np.round(rng.normal(np.where(label == 1, 46, 28), 9)).clip(8, 120).astype(int)
    num_dots    = rng.integers(1, 4, N) + label * rng.integers(0, 3, N)
    digit_ratio = np.clip(rng.normal(np.where(label == 1, 0.14, 0.04), 0.05, N), 0, 1).round(3)
    https       = (rng.random(N) < np.where(label == 1, 0.55, 0.86)).astype(int)
    age_days    = np.round(rng.lognormal(np.where(label == 1, 3.4, 6.4), 1.0)).astype(float).clip(0, 9000)

    age_days[rng.random(N) < 0.08] = -1
    reported_by_user = np.where(label == 1, rng.integers(1, 9, N), 0).astype(object)
    num_dots = num_dots.astype(object)
    num_dots[rng.integers(0, N)] = "unknown"

    df = pd.DataFrame({
        "url_length": url_length, "num_dots": num_dots, "digit_ratio": digit_ratio,
        "tld": tlds, "https": https, "age_days": age_days,
        "reported_by_user": reported_by_user, "label": label,
    })
    df = pd.concat([df, df.sample(118, random_state=7)], ignore_index=True)
    return df.sample(frac=1, random_state=1).reset_index(drop=True)


export = build_export()
export.head()

## Task 1: how many rows do you actually have?

`len()` is not the answer to this question.

In [ ]:
# TODO
rows_total  = len(export)
rows_unique = None      # how many are genuinely distinct?

print("rows in the file :", rows_total)
print("distinct rows    :", rows_unique)
print()
print("task 1 done?", rows_unique is not None and rows_unique < rows_total)

## Task 2: is every column the type it should be?

One of these columns is text and should not be. Find it, and find out why.

In [ ]:
# TODO: print the dtypes, then work out what is in the offending column
print(export.dtypes)

In [ ]:
suspect_column = None       # TODO: the column name
offending_value = None      # TODO: the value that made it text

print("suspect :", suspect_column)
print("value   :", offending_value)
print()
print("task 2 done?", suspect_column is not None and offending_value is not None)

## Task 3: which columns have missing values, and are they honest about it?

`isna()` finds the ones that admit it. At least one column is missing data
without admitting it, and lesson 1.6 is where you saw that trick.

In [ ]:
print("declared missing:")
print(export.isna().sum())

In [ ]:
# TODO: plot the distribution of every numeric column and look at the far left
numeric = export.select_dtypes(include="number").columns.tolist()
print("numeric columns:", numeric)

fig, axes = plt.subplots(1, len(numeric), figsize=(3.2 * len(numeric), 3))
for ax, col in zip(np.atleast_1d(axes), numeric):
    ax.hist(export[col], bins=40)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
hidden_missing_column = None    # TODO
placeholder_value     = None    # TODO
hidden_missing_count  = None    # TODO

print("column      :", hidden_missing_column)
print("placeholder :", placeholder_value)
print("how many    :", hidden_missing_count)
print()
print("task 3 done?", hidden_missing_count is not None and hidden_missing_count > 200)

## Task 4: what fraction is phishing, and what does that mean?

Then use lesson 1.8. Suppose you build a detector that catches 90% of phishing
and raises a false alarm on 3% of the rest. On a population with **this** base
rate, what fraction of its alerts would be real?

In [ ]:
base_rate = None      # TODO: fraction of rows that are phishing

# TODO: on 100,000 events at that base rate, with catches=0.90 and false alarm=0.03
tp = None
fp = None
precision = None

print("base rate :", base_rate)
print("precision :", precision)
print()
print("task 4 done?", precision is not None
      and 0 < precision < 1)

## Task 5: is any feature too good to be true?

This is the important one.

A feature that predicts the label almost perfectly is either the discovery of the
century or a mistake. It is a mistake. Group each candidate feature against the
label and look for one that separates them completely.

In [ ]:
# TODO: for each column that is not the label, how well does it split the label?
for col in ["tld", "https", "reported_by_user"]:
    print(f"--- {col} ---")
    print(export.groupby(col)["label"].agg(["size", "mean"]).round(3).head(10))
    print()

In [ ]:
leaking_feature = None      # TODO
why_it_leaks    = ""        # TODO: one sentence

print("leaking feature:", leaking_feature)
print("why            :", why_it_leaks)
print()
print("task 5 done?", leaking_feature is not None and len(why_it_leaks) > 20)

## Task 6: the obvious rule scores worse than doing nothing

Build the obvious rule, score it, and compare it against the laziest possible
baseline: predict "not phishing" for everything, every time.

You are going to find that the rule has **lower accuracy** than the baseline.
Decide which of the two you would rather deploy, and be able to say why.

In [ ]:
clean = export.drop_duplicates().copy()

lazy_accuracy = (clean["label"] == 0).mean()
print(f"predicting 'never phishing' scores {lazy_accuracy:.1%}")

In [ ]:
# TODO: a rule that does not use the leaking feature
rule = clean["tld"].isin(["tk", "xyz"])

accuracy  = (rule == clean["label"]).mean()
caught    = clean.loc[clean["label"] == 1, :].pipe(lambda d: rule[d.index].mean())
precision = clean.loc[rule, "label"].mean()

print(f"accuracy  {accuracy:.1%}")
print(f"recall    {caught:.1%}   of phishing caught")
print(f"precision {precision:.1%}   of alerts that are real")
print()
print(f"the lazy baseline scored {lazy_accuracy:.1%}, so accuracy alone tells you almost nothing")

In [ ]:
# TODO: a 95% interval on the rule's precision, using the bootstrap from lesson 1.9
alerts = clean.loc[rule, "label"].to_numpy()
rng = np.random.default_rng(0)

boot = None      # TODO: resample `alerts` 10,000 times and take the mean of each
lo = hi = None

print("precision            :", round(alerts.mean(), 4))
print("95% interval         :", lo, hi)
print()
print("task 6 done?", lo is not None)

## Write it up

The deliverable is not a clean dataset. It is a list, one entry per problem:

- what you found
- how you found it
- what you did about it
- what that cost

"Dropped 123 duplicate rows, found with `df.duplicated().sum()`, which removes 4%
of the training data" is a finding. "Cleaned the data" is not.

## Answers

Only after you have your own.

<details>
<summary>Task 1: rows</summary>

```python
rows_unique = len(export.drop_duplicates())
```

3,018 rows in the file, 2,895 distinct. 123 duplicates, which is 4% of the data
counted twice. Left in, they inflate any accuracy score and can put the same row
in both your training and test sets, which is the leakage that Track 2.3 is
about.

</details>

<details>
<summary>Task 2: types</summary>

```python
print(export["num_dots"].apply(type).value_counts())
print(export.loc[export["num_dots"].apply(lambda v: isinstance(v, str)), "num_dots"].unique())
```

`num_dots` is `object` because a single row holds the string `"unknown"`. One
value in three thousand turned a numeric column into text, and every numeric
comparison against it now fails. This is exactly the `dtypes` check from lesson
1.4, and it is why that check is worth doing every single time.

</details>

<details>
<summary>Task 3: hidden missing values</summary>

```python
hidden_missing_column = "age_days"
placeholder_value = -1
hidden_missing_count = int((export["age_days"] == -1).sum())
```

249 rows. `isna()` reports nothing, because -1 is a perfectly good number as far
as pandas is concerned. It shows up instantly in the histogram as a spike at the
far left, and it drags the mean down for everyone.

Turn it into a real missing value so nothing can average it by accident:

```python
export["age_days"] = export["age_days"].replace(-1, np.nan)
```

</details>

<details>
<summary>Task 4: base rate</summary>

```python
base_rate = clean["label"].mean()          # about 0.079

bad  = 100_000 * base_rate
good = 100_000 - bad
tp = bad * 0.90
fp = good * 0.03
precision = tp / (tp + fp)                  # about 0.72
```

At an 8% base rate a detector like that is actually usable, and about seven in
ten alerts are real. Now redo it at a base rate of 0.1%, which is closer to real
production traffic, and watch it fall to roughly 3%. Same detector.

**The base rate in your dataset is not the base rate in production**, because
datasets get balanced and production does not. That gap is the single most common
reason a model that tested well disappoints.

</details>

<details>
<summary>Task 5: the leak</summary>

`reported_by_user` is zero for every benign row and non-zero for every phishing
row. It predicts the label perfectly.

It leaks because of *when* it is recorded. A URL gets reported by a user after it
has been identified as phishing. The feature is not a property of the URL, it is
a record of the answer, and it cannot exist at the moment you would need to make
a prediction.

Train on it and you get a model with 100% accuracy that is worth nothing, because
in production the column is always zero for anything you have not already caught.

The general test: **could this value have been known before the thing you are
predicting happened?** If not, it is leakage. Track 2.3 is entirely about this.

</details>

<details>
<summary>Task 6: how sure are you</summary>

```python
idx  = rng.integers(0, len(alerts), size=(10_000, len(alerts)))
boot = alerts[idx].mean(axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
```

Predicting "never phishing" scores **91.9%**. The rule scores **80.8%**. By
accuracy, the rule is eleven points worse than a model that cannot detect
anything at all.

By any measure that matters, the rule is the better of the two. It raises 689
alerts and 184 of them are real, catching **78.3%** of the phishing in the
dataset at a precision of **26.7%**, plausibly 23.4% to 30.0%. The baseline
catches nothing, ever, and its 91.9% is just the fraction of the data that was
benign.

That is the whole problem with accuracy on imbalanced data, and it is why nobody
serious reports it. Track 2.7 replaces it with precision, recall and the
trade-off between them.

</details>

## Track 1 is done

You can load data from a file, a database or a query, check that it is what it
claims to be, find what is missing and what is quietly lying, plot it, reason
about how rare a thing is, and say how sure you are.

Track 2 starts building models. The reason this track came first is that a model
trained on the export you just examined would have scored 100% and been worthless,
and nothing about the model would have told you.